# Sample and spatially split Town Neck Beach

Select 8,122 labelled target cells from the prepared Dataset A descriptor, matching the number and split sizes of the current Svalbard beach-surface dataset. Preserve Town Neck's own six class proportions using largest-remainder allocation, not the Svalbard class labels or frequencies.

Outputs use the same four-column schema (date, cell_id, class, split) and basenames as the BS split CSVs, inside a separate bs_town_neck folder. The full raw dataset remains unchanged. No matrices are copied or normalized.

In [1]:
from pathlib import Path
import sys
import json
import hashlib
import pandas as pd
from IPython.display import display

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'Source').is_dir() and (p / 'Notebooks').is_dir())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_ROOT = PROJECT_ROOT / 'Data' / 'Raw' / 'bs_town_neck'
OUTPUT_ROOT = PROJECT_ROOT / 'Data' / 'Preprocessed' / 'bs_town_neck'
SPLIT_SIZES = {'train': 5386, 'validation': 1243, 'test': 1493}
RANDOM_SEED = 42
BLOCK_SIZE = 32
MAX_CONTEXT_RADIUS = 3

from Source.bs_town_neck_splits import create_sampled_splits, validate_splits
print(f'Output: {OUTPUT_ROOT}', flush=True)

Output: /Users/piotr/Praca/Nauka/publikacje/Context_Aware_Deep_Learning_Beach_Classification/software/Data/Preprocessed/bs_town_neck


## Spatial separation and sampling

Partition the 1 m cell grid into 32 x 32 m blocks. Assign whole blocks to training, validation, or testing using seed 42 and block counts proportional to requested sample sizes. Exclude the outermost three target rows/columns of every block (including partial outer blocks). Thus, every selected target's 7 x 7 radius-3 context stays within its own block.

Check block assignments for enough candidates to fulfil every class quota in every split. If needed, deterministically reshuffle blocks until this condition holds; no models or performance scores are involved. Then sample without replacement within each split and class. Retain the original raw cell IDs and original six class codes.

Explicitly verify that the unions of radius-3 context cells are disjoint across splits. The same targets and splits support all radii 0-3. Context RGB is read from the full raw dataset, but context-only cells are not labelled training examples.

This is block-wise evaluation within one survey, not a survey-date holdout. Dense Doodler labels are derived from sparse human annotations. Sampling preserves imbalance and reduces rare-class test counts; interpret class-specific metrics with that limitation.

**Training integration is separate:** configure the Town Neck raw root, split directory, Classes.xlsx and local grid geometry (559 rows x 454 columns). Existing Svalbard survey-date validation checks cannot be used unchanged. Compute normalization and missing-value replacement statistics from the training partition only.

In [2]:
provenance_path = OUTPUT_ROOT / 'bs_split_provenance.json'
if provenance_path.exists():
    provenance = json.loads(provenance_path.read_text())
    assert provenance['policy'] == 'town_neck_stratified_sample_spatial_blocks_v1'
    assert (RAW_ROOT / 'bs_town_neck_preprocessing_summary.json').exists()
    assert not (RAW_ROOT / 'bs_town_neck_preprocessing_in_progress.json').exists()
    assert provenance['requested_split_sizes'] == SPLIT_SIZES
    assert provenance['random_seed'] == RANDOM_SEED
    assert provenance['block_size_cells'] == BLOCK_SIZE
    assert provenance['max_context_radius'] == MAX_CONTEXT_RADIUS
    for name, expected in provenance['csv_sha256'].items():
        filename = 'bs_dataset_splits.csv' if name == 'all' else f'bs_{name}_split.csv'
        assert hashlib.sha256((OUTPUT_ROOT / filename).read_bytes()).hexdigest() == expected
    assert hashlib.sha256((RAW_ROOT / 'Descriptor_bs_town_neck.csv').read_bytes()).hexdigest() == provenance['raw_descriptor_sha256']
    assert hashlib.sha256((RAW_ROOT / 'bs_town_neck_grid_metadata.json').read_bytes()).hexdigest() == provenance['raw_grid_metadata_sha256']
    frames = {name: pd.read_csv(OUTPUT_ROOT / f'bs_{name}_split.csv', dtype={'date': str}) for name in SPLIT_SIZES}
    distribution = pd.read_csv(OUTPUT_ROOT / 'bs_town_neck_class_counts.csv').set_index('class')
    print('Existing sampled splits verified; no files overwritten.', flush=True)
else:
    frames, distribution, provenance = create_sampled_splits(RAW_ROOT, OUTPUT_ROOT, SPLIT_SIZES, RANDOM_SEED, BLOCK_SIZE, MAX_CONTEXT_RADIUS)
display(distribution)

Existing sampled splits verified; no files overwritten.


,class_name,train,validation,test,total
class,,,,,
1,Water,1207,279,335,1821
2,Sand,1301,300,360,1961
3,Gravel,75,17,21,113
4,Cobble/boulder,32,7,9,48
5,Vegetated,2699,623,748,4070
6,Development,72,17,20,109


In [3]:
source = pd.read_csv(RAW_ROOT / 'Descriptor_bs_town_neck.csv', dtype={'date': str})
grid = json.loads((RAW_ROOT / 'bs_town_neck_grid_metadata.json').read_text())
footprints, overlaps = validate_splits(frames, source, grid['grid_rows'], grid['grid_columns'], MAX_CONTEXT_RADIUS, SPLIT_SIZES)
combined = pd.read_csv(OUTPUT_ROOT / 'bs_dataset_splits.csv', dtype={'date': str})
pd.testing.assert_frame_equal(combined, pd.concat(frames.values(), ignore_index=True))
assert len(combined) == sum(SPLIT_SIZES.values())
actual_counts = pd.DataFrame({name: frame['class'].value_counts().sort_index() for name, frame in frames.items()}).astype(int)
actual_counts.index.name = 'class'
pd.testing.assert_frame_equal(distribution[list(SPLIT_SIZES)], actual_counts)
assert distribution['total'].equals(actual_counts.sum(axis=1).rename('total'))
assert distribution['class_name'].to_dict() == {int(k): v for k, v in grid['target_classes'].items()}
print('Total labelled targets:', len(combined), flush=True)
print('Split sizes:', {name: len(frame) for name, frame in frames.items()}, flush=True)
print('Radius-3 cross-split context overlap (must be zero):', overlaps, flush=True)
print('Raw RGB files remain in:', RAW_ROOT / 'beach_all' / '20160921', flush=True)

Total labelled targets: 8122


Split sizes: {'train': 5386, 'validation': 1243, 'test': 1493}


Radius-3 cross-split context overlap (must be zero): {'train/validation': 0, 'train/test': 0, 'validation/test': 0}


Raw RGB files remain in: /Users/piotr/Praca/Nauka/publikacje/Context_Aware_Deep_Learning_Beach_Classification/software/Data/Raw/bs_town_neck/beach_all/20160921
